In [ ]:
# Repository root (the folder with paths.py and 04_dmd_lqr_analysis.py), data root and figure folder (paths.py).
# Data are read from FAC_DATA_ROOT (default: repository root); figures are written to FAC_OUT_ROOT/figures/.
import sys
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "paths.py").is_file() and (p / "04_dmd_lqr_analysis.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Open this notebook inside the repository (figures/ or the repository root).")
sys.path.insert(0, str(REPO))
from paths import DATA, out

FIG_DIR = out("figures")
print("data:   ", DATA)
print("figures:", FIG_DIR)

In [ ]:
"""
SI Figures S4, S5, S7-S11 - Paper style
====================================

  S4: Surrogate Fidelity During Extreme Flows (1x3)
  S5: Surrogate Mass Fidelity (1x3)
  S7: Effort-Discharge Scaling (1x2)
  S8: Stream-Order Controllability (1x3) [representative member]
  S9: Robustness of Spatial Allocation (1x3)
  S10: Planning Resolution Flexibility (1x3 maps) [figures/figS10_huc_flexibility/make_figure.py]
  S11: Direction and Magnitude of Lifted Inputs (1x3)  [delta-based a,b; c from mass_conservation_summary.csv]
Figures S1-S3 and S6 are drawn by the scripts in figures/figS01_*, figS02_*, figS03_* and figS06_*.

Requires, under FAC_DATA_ROOT (paths.py):
  - outputs_flood_rl2/ (parquet results, includes nse_openloop and rlT_bias_openloop columns)
  - surrogate_fidelity_summary.csv, mass_conservation_summary.csv, ranking_stability.csv
  - outputs_flood_rl2/delta_atten_per_reach.parquet
  - stream_order_{pareto,summary}_VIC5_MPI-ESM1-2-HR_ssp245.csv
  - shape/Flowlines_huc02_12_Brazos.shp, shape/WBDHU12_Brazos.shp
  - huc_spatial_compact.csv and outputs_huc_control/ (S10)
"""

import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import linregress, spearmanr

SHP_PATH   = DATA / "shape/Flowlines_huc02_12_Brazos.shp"
RESULT_DIR = DATA / "outputs_flood_rl2"

GCMS = ["MPI-ESM1-2-HR","ACCESS-CM2","EC-Earth3",
        "CNRM-ESM2-1","BCC-CSM2-MR","MRI-ESM2-0","NorESM2-MM"]
SCENARIOS     = [126, 245, 370, 585]
TARGET_BUDGET = 10.0

SSP_LABELS = {126:"SSP1-2.6", 245:"SSP2-4.5", 370:"SSP3-7.0", 585:"SSP5-8.5"}
SSP_COLORS = {126:"#2d8e3d", 245:"#2166AC", 370:"#d4820a", 585:"#c72e29"}

MM = 1/25.4; W_FULL = 190*MM

# Paper style (NO dark theme)
mpl.rcParams.update({
    "font.family": "Arial", "font.size": 7,
    "axes.labelsize": 7, "axes.titlesize": 7, "axes.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.labelsize": 6, "ytick.labelsize": 6,
    "xtick.major.width": 0.4, "ytick.major.width": 0.4,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "xtick.direction": "out", "ytick.direction": "out",
    "legend.fontsize": 6, "legend.frameon": False,
    "legend.handlelength": 1.2, "legend.handletextpad": 0.4,
    "figure.dpi": 200, "savefig.dpi": 300,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42, "lines.linewidth": 0.8,
})

_RA  = 0.12; _MS  = 4.0; _MEW = 0.3; _GA  = 0.08
_PX  = -0.18; _PY  = 1.05
NSE_TH = 0.5
RETURN_PERIODS = [2, 5, 10, 20, 50, 100]

def lgrid(ax):
    ax.grid(True, alpha=_GA, lw=0.3, zorder=0); ax.set_axisbelow(True)

def plabel(ax, lab, x=_PX, y=_PY):
    ax.text(x, y, lab, transform=ax.transAxes,
            fontweight="bold", fontsize=9, va="top", ha="right")

def weighted_quantile(values, quantiles, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    m = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v, w = v[m], w[m]
    if len(v) == 0: return np.full_like(quantiles, np.nan, dtype=float)
    s = np.argsort(v); v, w = v[s], w[s]
    cw = np.cumsum(w); cdf = (cw - 0.5*w)/cw[-1]
    return np.interp(quantiles, cdf, v)

def flow_weighted_median(x, w):
    return float(weighted_quantile(x, [0.5], w)[0])

def jitter_strip(ax, x_pos, values, color, width=0.25, ms=3.5, alpha=0.5):
    jit = np.random.default_rng(42).uniform(-width/2, width/2, len(values))
    ax.scatter(x_pos + jit, values, s=ms, alpha=alpha, color=color,
               edgecolors="white", linewidths=0.15, zorder=3, rasterized=True)

def dot_whisker(ax, x_pos, values, color, lw=2.0, ms=7):
    med = np.median(values)
    q25, q75 = np.percentile(values, [25, 75])
    ax.plot([x_pos, x_pos], [q25, q75], color=color, lw=lw,
            solid_capstyle="round", zorder=4)
    ax.plot(x_pos, med, "o", color=color, ms=ms,
            markeredgecolor="black", markeredgewidth=0.5, zorder=5)
    return med

def add_scalebar(ax, gdf, km=100):
    b = gdf.total_bounds; bar = km / 96.0
    x0 = b[2] - bar - (b[2]-b[0])*0.04
    y0 = b[1] + (b[3]-b[1])*0.04
    ax.plot([x0, x0+bar], [y0, y0], "k-", lw=0.8, solid_capstyle="butt", zorder=10)
    ax.text(x0+bar/2, y0+(b[3]-b[1])*0.015, f"{km} km",
            ha="center", fontsize=5, zorder=10)

def interpolate_to_budget(grp, target=TARGET_BUDGET):
    eff_by_r = grp.groupby("R_weight")["eff_total_L1"].sum()/1e9
    r_values, efforts = eff_by_r.index.values, eff_by_r.values
    if len(r_values) < 2:
        return grp[grp["R_weight"]==r_values[0]].copy()
    si = np.argsort(efforts); es, rs = efforts[si], r_values[si]
    if target<=es[0]:  return grp[grp["R_weight"]==rs[0]].copy()
    if target>=es[-1]: return grp[grp["R_weight"]==rs[-1]].copy()
    ia = np.searchsorted(es, target); ib = ia-1
    w = 0.5 if es[ia]==es[ib] else (target-es[ib])/(es[ia]-es[ib])
    dlo = grp[grp["R_weight"]==rs[ib]].set_index("reach_id")
    dhi = grp[grp["R_weight"]==rs[ia]].set_index("reach_id")
    common = dlo.index.intersection(dhi.index)
    dlo, dhi = dlo.loc[common], dhi.loc[common]
    out = dlo.copy()
    for c in dlo.select_dtypes(include=[np.number]).columns:
        if c in ("R_weight","scenario","reach_id"): continue
        out[c] = dlo[c]*(1-w)+dhi[c]*w
    return out.reset_index()


# DATA LOADING (memory-streamed: no global df_all)
import gc
print("Loading data ...")
gdf_net = gpd.read_file(SHP_PATH)
gdf_net["COMID"] = gdf_net["COMID"].astype(int)

# Stream results parquets one at a time and accumulate two small derived
# frames: df_sel (interpolated to TARGET_BUDGET per member) and df_base
# (minimum R_weight per member). The full long-format results table
# (~5 GB of parquet) is never held in memory.
print(f"  Streaming results parquets and interpolating to {TARGET_BUDGET} Gm3/yr ...")
sel_chunks = []
base_chunks = []
for fp in sorted(RESULT_DIR.glob("results_*.parquet")):
    df = pd.read_parquet(fp)
    if "hydro" not in df.columns:
        df["hydro"] = "VIC5"
    df["reach_id"] = df["reach_id"].astype(int)
    df["scenario"] = df["scenario"].astype(int)

    for k, g in df.groupby(["gcm","scenario","hydro"]):
        sel_chunks.append(interpolate_to_budget(g))

    idx_min = df.groupby(["gcm","hydro","scenario"])["R_weight"].transform("min") == df["R_weight"]
    base_chunks.append(
        df[idx_min].drop_duplicates(subset=["gcm","hydro","scenario","reach_id"]).copy()
    )
    del df, idx_min
    gc.collect()

df_sel = pd.concat(sel_chunks, ignore_index=True)
df_base = pd.concat(base_chunks, ignore_index=True)
del sel_chunks, base_chunks
gc.collect()

mc = [c for c in ["vol_hi_baseline","vol_hi_controlled","eff_total_L1",
                   "residual_ratio","mean_flow_ref"] if c in df_sel.columns]
df_ens = df_sel.groupby(["reach_id","scenario"])[mc].median().reset_index()
df_ens = df_ens.merge(gdf_net[["COMID","StreamOrde","TotDASqKM"]].rename(
    columns={"COMID":"reach_id"}), on="reach_id", how="left")
if "mean_flow_ref" in df_sel.columns and "mean_flow_ref" not in df_ens.columns:
    df_ens["mean_flow_ref"] = df_ens["reach_id"].map(
        df_sel.groupby("reach_id")["mean_flow_ref"].first())
df_ens["so"] = df_ens["StreamOrde"].astype(float).astype("Int64")

daily_frames = [pd.read_parquet(f) for f in sorted(RESULT_DIR.glob("daily_*.parquet"))]
df_daily = pd.concat(daily_frames, ignore_index=True) if daily_frames else None
if df_daily is not None:
    df_daily = df_daily[df_daily["budget_target"]==TARGET_BUDGET]
    df_daily["date"] = pd.to_datetime(df_daily["date"])
    df_daily["year"] = df_daily["date"].dt.year

if "atten_fraction" not in df_base.columns and "eff_atten_L1" in df_base.columns:
    df_base["atten_fraction"] = df_base["eff_atten_L1"]/(df_base["eff_total_L1"]+1e-12)

print(f"  {df_ens['reach_id'].nunique():,} reaches")


In [ ]:
# FIG S4 - Surrogate Fidelity During Extreme Flows (1x3)
#   (a) Per-reach RL10 bias histogram         [from outputs_flood_rl2/ parquet]
#   (b) Basin-aggregate hydrograph RL bias    [from surrogate_fidelity_summary.csv]
#   (c) Stream-order aggregate RL10 bias      [from surrogate_fidelity_summary.csv]
#
# Panel (a) reads the per-reach `rl10_bias_openloop` column directly from the
# step-4 parquets (already loaded as df_base by the data-loading cell). Panels (b) and (c)
# need the basin/order-aggregate AMS, which can only be computed by re-running
# the surrogate (Jensen's inequality on the AMS-RL operator means that the
# flow-weighted reach-average is NOT equal to the basin-aggregate RL bias).
# Those values are pre-computed offline by
# `analyses/surrogate_fidelity/si_surrogate_fidelity.py` and
# saved to surrogate_fidelity_summary.csv (one row per member, ~10 KB).
print("\nFIG S4 - Surrogate Fidelity")

RPS = [2, 5, 10, 20, 50, 100]
NSE_TH_VAL = 0.5

df_val = df_base.copy()
if "nse_openloop" in df_val.columns:
    # reaches with NSE >= 0.5 and a positive reference RL10 (the relative bias needs RL10 > 0)
    df_val = df_val[(df_val["nse_openloop"] >= NSE_TH_VAL) & (df_val["rl10_true"] > 1e-6)]
print(f"  (a) {len(df_val):,} reach-member rows (NSE >= {NSE_TH_VAL}, reference RL10 > 0)")

if "StreamOrde" not in df_val.columns:
    df_val = df_val.merge(
        gdf_net[["COMID","StreamOrde"]].rename(columns={"COMID":"reach_id"}),
        on="reach_id", how="left")

fidelity_path = DATA / "surrogate_fidelity_summary.csv"
if fidelity_path.exists():
    df_fid = pd.read_csv(fidelity_path)
    print(f"  (b,c) {len(df_fid)} members from {fidelity_path}")
else:
    df_fid = None
    print(f"  (b,c) {fidelity_path} not found - run `python analyses/surrogate_fidelity/si_surrogate_fidelity.py` first")

np.random.seed(42)
fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 65*MM),
                          gridspec_kw={"wspace": 0.38})

# (a) Per-reach RL10 bias histogram
ax = axes[0]
if "rl10_bias_openloop" in df_val.columns:
    bias_arr = pd.to_numeric(df_val["rl10_bias_openloop"], errors="coerce").dropna().values
    bias_arr = bias_arr[np.isfinite(bias_arr)]
    ax.hist(bias_arr, bins=80, range=(-1, 1),
            color="#9ecae1", edgecolor="#4292c6", linewidth=0.3,
            density=True, zorder=2)
    med = np.median(bias_arr)
    ax.axvline(med, color="k", ls="--", lw=0.8, zorder=3)
    ax.axvline(0, color="#999", ls="-", lw=0.4, zorder=1)
    ymax = ax.get_ylim()[1]
    ax.text(med - 0.03, ymax * 0.92,
            f"median\n{med:.3f}", fontsize=5.5, ha="right", va="top")
    print(f"  (a) RL10 bias median = {med:.4f}")
ax.set_xlabel(r"RL$_{10}$ relative bias")
ax.set_ylabel("Density")
lgrid(ax); plabel(ax, "a", x=-0.10, y=1.1)

# (b) Basin-aggregate hydrograph RL bias by return period
ax = axes[1]
if df_fid is not None and any(f"basin_rl{T}_bias" in df_fid.columns for T in RPS):
    bias_mat = np.array([
        [df_fid[f"basin_rl{T}_bias"].iloc[i] if f"basin_rl{T}_bias" in df_fid.columns else np.nan
         for T in RPS]
        for i in range(len(df_fid))
    ]) * 100  # to %

    log_rps = np.log10(RPS)
    for i in range(bias_mat.shape[0]):
        jitter = np.random.uniform(-0.06, 0.06, len(RPS))
        x_jit = 10 ** (log_rps + jitter)
        ax.scatter(x_jit, bias_mat[i, :],
                   s=5, color="#9ecae1", alpha=0.35, zorder=1,
                   edgecolors="none")
    med_line = np.nanmedian(bias_mat, axis=0)
    ax.plot(RPS, med_line, "o-", color="#d62728",
            ms=5, lw=1.0, zorder=3, label="Ensemble median")
    ax.axhline(0, color="#999", ls="-", lw=0.4, zorder=0)
    ax.axhline(5, color="#999", ls=":", lw=0.5, zorder=0)
    ax.axhline(-5, color="#999", ls=":", lw=0.5, zorder=0)
    ax.set_ylim(-10, 10)
    ax.text(0.97, 0.05, f"n = {bias_mat.shape[0]} members",
            transform=ax.transAxes, fontsize=5, va="bottom", ha="right",
            color="#555")
    print(f"  (b) median basin RL10 bias = {med_line[2]:+.2f}%")
else:
    ax.text(0.5, 0.5,
            "surrogate_fidelity_summary.csv\nnot found.\n\n"
            "Run `python analyses/surrogate_fidelity/\n"
            "si_surrogate_fidelity.py`\n"
            "to generate it.",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=6, color="#666")

ax.set_xlabel("Return period (years)")
ax.set_ylabel("Basin-total RL bias (%)")
ax.set_xscale("log")
ax.set_xticks(RPS)
ax.set_xticklabels([str(r) for r in RPS])
ax.legend(loc="lower left", fontsize=5, handlelength=2)
lgrid(ax); plabel(ax, "b", x=-0.10, y=1.1)

# (c) Stream-order aggregate RL10 bias
ax = axes[2]
if df_fid is not None:
    order_cols = sorted(
        [c for c in df_fid.columns if c.startswith("order") and c.endswith("_rl10_bias")],
        key=lambda c: int(c.replace("order", "").replace("_rl10_bias", ""))
    )
    if order_cols:
        orders = [int(c.replace("order", "").replace("_rl10_bias", "")) for c in order_cols]
        order_mat = df_fid[order_cols].values * 100  # to %

        x_pos = np.arange(len(orders))
        for i in range(order_mat.shape[0]):
            jitter = np.random.uniform(-0.15, 0.15, len(orders))
            ax.scatter(x_pos + jitter, order_mat[i, :],
                       s=5, color="#9ecae1", alpha=0.35, zorder=1,
                       edgecolors="none")
        med_o = np.nanmedian(order_mat, axis=0)
        ax.plot(x_pos, med_o, "o-", color="#d62728",
                ms=5, lw=1.0, zorder=3, label="Ensemble median")
        ax.axhline(0, color="#999", ls="-", lw=0.4, zorder=0)
        ax.axhline(5, color="#999", ls=":", lw=0.5, zorder=0)
        ax.axhline(-5, color="#999", ls=":", lw=0.5, zorder=0)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([str(o) for o in orders])
        ax.legend(loc="upper right", fontsize=5, handlelength=2)
        print(f"  (c) {len(orders)} stream orders")
else:
    ax.text(0.5, 0.5, "surrogate_fidelity_summary.csv\nnot found",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=6, color="#666")

ax.set_ylim(-10, 10)
ax.set_xlabel("Stream order")
ax.set_ylabel(r"RL$_{10}$ bias (%)")
lgrid(ax); plabel(ax, "c", x=-0.10, y=1.1)

for ext in ("png", "pdf"):
    fig.savefig(FIG_DIR / f"figS4_extreme_flows.{ext}")
plt.show()
print("  Saved figS4_extreme_flows")


In [ ]:
# FIG S5 - Surrogate Mass Fidelity (1x3)
print("\nFIG S5 - Surrogate Mass Fidelity")

df_mass = pd.read_csv(DATA / "mass_conservation_summary.csv")
print(f"  Loaded {len(df_mass)} members")
scenarios_s5 = sorted(df_mass["ssp"].unique())

fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 70*MM),
                          gridspec_kw={"wspace": 0.36})

# (a) Volume ratio
ax = axes[0]
ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=1)
for i, scn in enumerate(scenarios_s5):
    vals = df_mass[df_mass["ssp"]==scn]["vol_ratio"].dropna().values
    jitter_strip(ax, i, vals, SSP_COLORS[scn], ms=4, alpha=0.55)
    dot_whisker(ax, i, vals, SSP_COLORS[scn])
all_vr = df_mass["vol_ratio"].dropna().values
ax.text(0.97, 0.05, f"all: {np.median(all_vr):.3f}\n(n={len(all_vr)})",
        transform=ax.transAxes, fontsize=6.5, va="bottom", ha="right",
        bbox=dict(boxstyle="round,pad=0.15", fc="white",
                  ec="none", lw=0.3, alpha=0.85))
ax.set_xticks(range(len(scenarios_s5)))
ax.set_xticklabels([SSP_LABELS[s] for s in scenarios_s5], fontsize=5.5)
ax.set_ylabel("Volume ratio\n(surrogate / reference)")
ax.set_xlim(-0.5, len(scenarios_s5)-0.5)
ax.set_ylim(0.95, 1.05)
lgrid(ax); plabel(ax, "a")

# (b) Basin-total delta correlation r
ax = axes[1]
ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=1)
for i, scn in enumerate(scenarios_s5):
    vals = df_mass[df_mass["ssp"]==scn]["delta_corr"].dropna().values
    jitter_strip(ax, i, vals, SSP_COLORS[scn], ms=4, alpha=0.55)
    dot_whisker(ax, i, vals, SSP_COLORS[scn])
all_r  = df_mass["delta_corr"].dropna().values
ax.text(0.97, 0.05, f"all: {np.median(all_r):.3f}\n(n={len(all_r)})",
        transform=ax.transAxes, fontsize=6.5, va="bottom", ha="right",
        bbox=dict(boxstyle="round,pad=0.15", fc="white",
                  ec="none", lw=0.3, alpha=0.85))
ax.set_xticks(range(len(scenarios_s5)))
ax.set_xticklabels([SSP_LABELS[s] for s in scenarios_s5], fontsize=5.5)
ax.set_ylabel(r"Basin-total $\delta$ correlation")
ax.set_xlim(-0.5, len(scenarios_s5)-0.5)
ax.set_ylim(-0.05, 1.05)
lgrid(ax); plabel(ax, "b")

# (c) Per-reach delta correlation median
ax = axes[2]
ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=1)
for i, scn in enumerate(scenarios_s5):
    vals = df_mass[df_mass["ssp"]==scn]["reach_median"].dropna().values
    jitter_strip(ax, i, vals, SSP_COLORS[scn], ms=4, alpha=0.55)
    dot_whisker(ax, i, vals, SSP_COLORS[scn])
all_rm = df_mass["reach_median"].dropna().values
ax.text(0.97, 0.05, f"all: {np.median(all_rm):.3f}\n(n={len(all_rm)})",
        transform=ax.transAxes, fontsize=6.5, va="bottom", ha="right",
        bbox=dict(boxstyle="round,pad=0.15", fc="white",
                  ec="none", lw=0.3, alpha=0.85))
ax.set_xticks(range(len(scenarios_s5)))
ax.set_xticklabels([SSP_LABELS[s] for s in scenarios_s5], fontsize=5.5)
ax.set_ylabel("Per-reach $\\delta$ correlation\n(median across reaches)")
ax.set_xlim(-0.5, len(scenarios_s5)-0.5)
ax.set_ylim(-0.05, 1.05)
lgrid(ax); plabel(ax, "c")

for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS5_mass_fidelity.{ext}")
plt.show()


In [ ]:
# FIG S7 - Effort-Discharge Scaling (1x2)
print("\nFIG S7 - Effort scaling")
sub = df_ens[df_ens["scenario"]==245].copy()
eff = sub["eff_total_L1"].to_numpy(float)
da  = sub["TotDASqKM"].to_numpy(float)
mf  = sub["mean_flow_ref"].to_numpy(float) if "mean_flow_ref" in sub.columns else None

fig, axes = plt.subplots(1, 2, figsize=(W_FULL, 70*MM),
                          gridspec_kw={"wspace":0.28})

panels = [
    (axes[0], mf,  "Blues",
     r"Mean annual flow $\overline{Q}$ (log$_{10}$ m$^3$ s$^{-1}$)", "a"),
    (axes[1], da,  "Greens",
     r"Drainage area (log$_{10}$ km$^2$)", "b"),
]

for ax, xv, cmap_name, xlab, lab in panels:
    if xv is None:
        ax.text(0.5,0.5,"Data not available",transform=ax.transAxes,ha="center")
        plabel(ax,lab); continue

    m = (eff>0)&(xv>0)&np.isfinite(eff)&np.isfinite(xv)
    lx, ly = np.log10(xv[m]), np.log10(eff[m])
    lr = linregress(lx, ly)

    ax.hexbin(lx, ly, gridsize=45, cmap=cmap_name, mincnt=1,
              linewidths=0, alpha=0.75, zorder=1, rasterized=True)

    xf = np.array([lx.min(), lx.max()])
    ax.plot(xf, lr.slope*xf+lr.intercept, "--", color="#333",
            lw=1.3, zorder=5)

    ax.text(0.05, 0.95,
            f"slope = {lr.slope:.2f}\n$R^2$ = {lr.rvalue**2:.2f}",
            transform=ax.transAxes, fontsize=6.5, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.2", fc="white",
                      ec="none", lw=0.4, alpha=0.92))

    ax.set_xlabel(xlab)
    ax.set_ylim(-1, 8)
    if lab=="a": ax.set_ylabel(r"Annual effort (log$_{10}$ m$^3$ yr$^{-1}$)")
    lgrid(ax); plabel(ax, lab)
    print(f"  ({lab}) R2={lr.rvalue**2:.3f}")

for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS7_effort_scaling.{ext}")
plt.show()


In [ ]:
# FIG S8 - Stream-Order Controllability (1x3) [representative member: VIC5/MPI-ESM1-2-HR/SSP2-4.5]
#   (a) Brazos river network colored by Strahler stream order (reach counts per order from the summary table)
#   (b) Flood-volume reduction against effort with control restricted to order \u2264 N
#   (c) Marginal contribution of each stream order to flood-volume reduction at fixed efforts
# Source: analyses/stream_order/dmd_lqr_stream_order.py outputs, under FAC_DATA_ROOT:
#   stream_order_summary_VIC5_MPI-ESM1-2-HR_ssp245.csv (R-sweep at fine grid)
#   stream_order_pareto_VIC5_MPI-ESM1-2-HR_ssp245.csv  (R-sweep at coarse grid; used if the summary is missing)
print("\nFIG S8 - Stream-Order Controllability")
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.collections import LineCollection
import matplotlib.colors as mcolors

SUMMARY_CSV = DATA / "stream_order_summary_VIC5_MPI-ESM1-2-HR_ssp245.csv"
if not SUMMARY_CSV.exists():
    SUMMARY_CSV = DATA / "stream_order_pareto_VIC5_MPI-ESM1-2-HR_ssp245.csv"
EFFORT_TARGETS = [1, 3, 5, 10, 15]
ORDER_COLORS = {1: "#c6dbef", 2: "#9ecae1", 3: "#6baed6", 4: "#3182bd", 5: "#08519c", 6: "#78419a", 7: "#4a1486"}
ORDER_LW_MAP = {0: 0.08, 1: 0.12, 2: 0.22, 3: 0.55, 4: 0.95, 5: 1.30, 6: 1.60, 7: 2.20}

def _segs(geom):
    if geom is None or geom.is_empty: return []
    if geom.geom_type == "LineString": return [np.asarray(geom.coords)]
    if geom.geom_type == "MultiLineString": return [np.asarray(g.coords) for g in geom.geoms]
    return []

def draw_net_rgba(ax, gdf, colors, lw, alpha=0.92):
    segs, cols, lws = [], [], []
    for geom, c, w in zip(gdf.geometry.values, colors, lw):
        for part in _segs(geom):
            segs.append(part); cols.append(c); lws.append(float(w))
    lc = LineCollection(segs, colors=cols, linewidths=lws, capstyle="round", joinstyle="round", alpha=alpha, zorder=1)
    lc.set_rasterized(True); ax.add_collection(lc)

def plabel8(ax, lab, x=-0.08, y=1.05):
    ax.text(x, y, lab, transform=ax.transAxes, fontweight="bold", fontsize=9, va="top", ha="right")

def lgrid8(ax, alpha=0.10, lw=0.3):
    ax.grid(True, alpha=alpha, lw=lw, zorder=0); ax.set_axisbelow(True)

df = pd.read_csv(SUMMARY_CSV)
labels_order = sorted(df["constraint_label"].unique(), key=lambda s: int(s.split("_")[-1]))
n_orders = len(labels_order)
orders = [int(l.split("_")[-1]) for l in labels_order]

def _interp_red(sub, target):
    sub = sub.sort_values("effort_Gm3")
    above = sub[sub["effort_Gm3"] >= target].head(1)
    below = sub[sub["effort_Gm3"] <= target].tail(1)
    if len(above) == 0: return below.iloc[0]["reduction_frac"] * 100
    if len(below) == 0: return above.iloc[0]["reduction_frac"] * 100
    a, b = above.iloc[0], below.iloc[0]
    if abs(a["effort_Gm3"] - b["effort_Gm3"]) < 1e-6: return a["reduction_frac"] * 100
    w = (target - b["effort_Gm3"]) / (a["effort_Gm3"] - b["effort_Gm3"])
    return (b["reduction_frac"] + w * (a["reduction_frac"] - b["reduction_frac"])) * 100

cumul = {eff: [_interp_red(df[df["constraint_label"] == l], eff) for l in labels_order] for eff in EFFORT_TARGETS}
marginals = {eff: [c[0]] + [max(c[i] - c[i-1], 0) for i in range(1, n_orders)] for eff, c in cumul.items()}
reach_counts_cum = {int(l.split("_")[-1]): int(df[df["constraint_label"] == l].iloc[0]["n_ctrl_reaches"]) for l in labels_order}
counts_per_order = {}; prev = 0
for o in orders:
    counts_per_order[o] = reach_counts_cum[o] - prev; prev = reach_counts_cum[o]
total_reaches = reach_counts_cum[orders[-1]]
curves_by_label = {}
for label in labels_order:
    sub = df[df["constraint_label"] == label].sort_values("effort_Gm3")
    curves_by_label[label] = (sub["effort_Gm3"].values, sub["reduction_frac"].values * 100)
print(f"  Orders: {orders}, total reaches: {total_reaches}, per order: {counts_per_order}")

fig = plt.figure(figsize=(W_FULL, 80*MM))
gs = GridSpec(1, 3, figure=fig, width_ratios=[1.0, 0.85, 0.85], wspace=0.30, left=0.02, right=0.97, bottom=0.14, top=0.95)
ax_a = fig.add_subplot(gs[0, 0]); ax_b = fig.add_subplot(gs[0, 1]); ax_c = fig.add_subplot(gs[0, 2])
_bnds = gdf_net.total_bounds
_xp = (_bnds[2]-_bnds[0])*0.02; _yp = (_bnds[3]-_bnds[1])*0.02
ax_a.set_aspect("equal"); ax_a.axis("off")
gdf_net.plot(ax=ax_a, color="#f0f0f0", linewidth=0.02, alpha=0.22, zorder=0)
subset0 = gdf_net[gdf_net["StreamOrde"] == 0]
if len(subset0) > 0:
    draw_net_rgba(ax_a, subset0, np.array([mcolors.to_rgba("#E8E8E8")] * len(subset0)), np.full(len(subset0), 0.08), alpha=0.40)
for order in sorted(ORDER_COLORS):
    subset = gdf_net[gdf_net["StreamOrde"] == order]
    if len(subset) == 0: continue
    draw_net_rgba(ax_a, subset, np.array([mcolors.to_rgba(ORDER_COLORS[order])] * len(subset)),
                  np.full(len(subset), ORDER_LW_MAP.get(order, 0.1)), alpha=0.90)
ax_a.set_xlim(_bnds[0]-_xp, _bnds[2]+_xp); ax_a.set_ylim(_bnds[1]-_yp, _bnds[3]+_yp)
add_scalebar(ax_a, gdf_net)
plabel8(ax_a, "a", x=-0.12, y=1.06)
ax_a.legend(handles=[Line2D([0], [0], color=ORDER_COLORS[o], lw=max(ORDER_LW_MAP[o]*1.5, 1.0),
                            label=f"Order {o} ({counts_per_order[o]:,}, {counts_per_order[o]/total_reaches*100:.0f}%)")
                     for o in orders],
            loc="lower left", fontsize=5.5, frameon=False, title="Stream order (reaches, %)", title_fontsize=6)

for i, label in enumerate(labels_order):
    o = orders[i]
    if o in (1, orders[-1]): continue
    eff, red = curves_by_label[label]
    ax_b.plot(eff, red, color=ORDER_COLORS[o], lw=0.6, ls="-", alpha=0.35)
eff1, red1 = curves_by_label[labels_order[0]]; effA, redA = curves_by_label[labels_order[-1]]
x_common = np.linspace(0.3, min(eff1.max(), effA.max(), 16), 200)
ax_b.fill_between(x_common, np.interp(x_common, eff1, red1), np.interp(x_common, effA, redA), color=ORDER_COLORS[orders[-1]], alpha=0.08)
ax_b.plot(eff1, red1, color=ORDER_COLORS[1], lw=2.0, alpha=0.95, zorder=5)
ax_b.plot(effA, redA, color=ORDER_COLORS[orders[-1]], lw=2.0, alpha=0.95, zorder=5)
ax_b.set_xlabel(r"Effort (Gm$^3$ yr$^{-1}$)"); ax_b.set_ylabel("Flood volume reduction (%)")
ax_b.set_xlim(0, 16); ax_b.set_ylim(0, 95); lgrid8(ax_b, alpha=0.08)
share1 = counts_per_order[orders[0]] / total_reaches * 100
ax_b.legend(handles=[Line2D([0], [0], color=ORDER_COLORS[1], lw=2.0, label=f"$\\leq$1 ({share1:.0f}%)")]
                    + [Line2D([0], [0], color=ORDER_COLORS[o], lw=0.6, alpha=0.5, label=f"$\\leq${o}") for o in orders[1:-1]]
                    + [Line2D([0], [0], color=ORDER_COLORS[orders[-1]], lw=2.0, label="All (100%)")],
            loc="lower right", fontsize=5, ncol=2, frameon=False, title="Max order included", title_fontsize=5.5,
            columnspacing=0.8, handlelength=1.5)
plabel8(ax_b, "b", x=-0.12, y=1.06)

x = np.arange(len(EFFORT_TARGETS)); bottoms = np.zeros(len(EFFORT_TARGETS))
for order in orders:
    vals = np.array([marginals[eff][order - orders[0]] for eff in EFFORT_TARGETS])
    ax_c.bar(x, vals, 0.55, bottom=bottoms, color=ORDER_COLORS[order], edgecolor="white", linewidth=0.3)
    if order == orders[0]:
        for i, v in enumerate(vals):
            ax_c.text(x[i], bottoms[i] + v/2, f"{v:.0f}%", ha="center", va="center", fontsize=5.5, color="white", fontweight="bold")
    bottoms += vals
for i in range(len(EFFORT_TARGETS)):
    ax_c.text(x[i], bottoms[i] + 0.8, f"{bottoms[i]:.0f}%", ha="center", va="bottom", fontsize=5.5, color="#444")
    print(f"  (c) E={EFFORT_TARGETS[i]:>2} Gm3/yr: total={bottoms[i]:5.1f}%, order-1={marginals[EFFORT_TARGETS[i]][0]:5.1f}%")
ax_c.set_xticks(x); ax_c.set_xticklabels([f"{e}" for e in EFFORT_TARGETS])
ax_c.set_xlabel(r"Effort (Gm$^3$ yr$^{-1}$)"); ax_c.set_ylabel("Flood volume reduction (%)")
ax_c.set_ylim(0, 100); lgrid8(ax_c, alpha=0.08)
ax_c.legend(handles=[Patch(facecolor=ORDER_COLORS[o], edgecolor="white", linewidth=0.3,
                           label=f"Order {o} ({counts_per_order[o]/total_reaches*100:.0f}%)") for o in orders],
            loc="upper left", fontsize=4.5, ncol=2, frameon=False, title="Marginal contribution", title_fontsize=5)
plabel8(ax_c, "c", x=-0.12, y=1.06)
for ext in ("png", "pdf"): fig.savefig(FIG_DIR / f"figS8_stream_order.{ext}")
plt.show()
print("  Saved figS8_stream_order")

In [ ]:
# FIG S9 - Robustness (1x3)
print("\nFIG S9 - Robustness")
so_groups = [("1\u20132",[1,2]), ("3\u20134",[3,4]), ("5+",[5,6,7,8,9])]

fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 70*MM),
                          gridspec_kw={"wspace":0.38})

# (a) Stream-order effort
ax = axes[0]
x_pos = np.arange(len(so_groups))
bw = 0.18
for i, scn in enumerate(SCENARIOS):
    sub = df_ens[df_ens["scenario"]==scn]
    total = sub["eff_total_L1"].sum()
    fracs = [sub[sub["so"].isin(o)]["eff_total_L1"].sum()/total*100
             for _,o in so_groups]
    ax.bar(x_pos+(i-1.5)*bw, fracs, bw,
           color=SSP_COLORS[scn], alpha=0.75, edgecolor="black",
           linewidth=0.3, label=SSP_LABELS[scn])

ax.set_xticks(x_pos)
ax.set_xticklabels([g for g,_ in so_groups])
ax.set_xlabel("Stream order")
ax.set_ylabel("Share of basin effort (%)")
ax.set_ylim(0, 58)
ax.legend(fontsize=4.5, loc="upper left", ncol=2,
          handlelength=0.8, columnspacing=0.5)
lgrid(ax); plabel(ax,"a")

for scn in SCENARIOS:
    sub = df_ens[df_ens["scenario"]==scn]
    t = sub["eff_total_L1"].sum()
    o5 = sub[sub["so"].isin([5,6,7,8,9])]["eff_total_L1"].sum()/t*100
    print(f"  (a) {SSP_LABELS[scn]}: Order 5+ = {o5:.1f}%")

# (b) Temporal late/early ratio
ax = axes[1]
if df_daily is not None:
    df_daily["period"] = np.where(df_daily["year"]<2060, "early", "late")
    grp = ["scenario","gcm"] + (["hydro"] if "hydro" in df_daily.columns else [])
    merge_on = ["gcm"] + (["hydro"] if "hydro" in df_daily.columns else [])

    pb = (df_daily.groupby(grp+["period"])
          .agg(cl=("basin_exceed_controlled","sum")).reset_index())
    pb["cl_Gm3"] = pb["cl"]/1e9

    for i, scn in enumerate(SCENARIOS):
        e = pb[(pb["scenario"]==scn)&(pb["period"]=="early")].set_index(merge_on)["cl_Gm3"]
        l = pb[(pb["scenario"]==scn)&(pb["period"]=="late")].set_index(merge_on)["cl_Gm3"]
        c = e.index.intersection(l.index)
        vals = (l.loc[c]/(e.loc[c]+1e-12)).values

        med  = np.median(vals)
        q25, q75 = np.percentile(vals, [25, 75])
        ax.plot([i, i], [q25, q75], color=SSP_COLORS[scn], lw=2.0,
                solid_capstyle="round", zorder=3)
        ax.plot(i, med, "o", color=SSP_COLORS[scn], ms=7,
                markeredgecolor="black", markeredgewidth=0.6, zorder=4)
        ax.text(i, q75+0.08, f"{med:.2f}", ha="center", fontsize=5.5,
                fontweight="bold")
        print(f"  (b) {SSP_LABELS[scn]}: {med:.2f}")

    ax.axhline(1.0, color="#777", ls="--", lw=0.6, zorder=0)
    ax.set_xticks(range(len(SCENARIOS)))
    ax.set_xticklabels([SSP_LABELS[s] for s in SCENARIOS], fontsize=5.5)
    ax.set_ylabel("Late / early century\nexceedance ratio")
    ax.set_xlim(-0.5, len(SCENARIOS)-0.5)
    ax.set_ylim(0.4, 1.8)
else:
    ax.text(0.5,0.5,"Daily data\nnot available",transform=ax.transAxes,
            ha="center",fontsize=7)
lgrid(ax); plabel(ax,"b")

# (c) Ranking stability  [pre-computed from the full results by
#     analyses/ranking_stability/si_ranking_stability.py: ranking_stability.csv, one row per member]
ax = axes[2]
ranking_path = DATA / "ranking_stability.csv"
if ranking_path.exists():
    C = pd.read_csv(ranking_path)
    C["scenario"] = C["scenario"].astype(int)
    print(f"  (c) {len(C)} members from {ranking_path}")
else:
    C = pd.DataFrame()
    print(f"  (c) {ranking_path} not found - run `python analyses/ranking_stability/si_ranking_stability.py`")

if len(C) > 0:
    offset = 0.12
    for i, scn in enumerate(SCENARIOS):
        cs = C[C["scenario"]==scn]
        for j, (metric, color, marker, label) in enumerate([
            ("spearman", "#4292c6", "o", r"Spearman $\rho$"),
            ("overlap",  "#E6550D", "s", "Top-10% overlap"),
        ]):
            vals = cs[metric].values
            med  = np.median(vals)
            q25, q75 = np.percentile(vals, [25, 75])
            xpos = i + (j-0.5)*offset*2
            ax.plot([xpos, xpos], [q25, q75], color=color, lw=1.5,
                    solid_capstyle="round", zorder=3)
            ax.plot(xpos, med, marker, color=color, ms=6,
                    markeredgecolor="black", markeredgewidth=0.5,
                    zorder=4, label=label if i==0 else "")

    all_v = np.concatenate([C["spearman"].values, C["overlap"].values])
    ax.set_ylim(0.5, 1.02)
    ax.set_xticks(range(len(SCENARIOS)))
    ax.set_xticklabels([SSP_LABELS[s] for s in SCENARIOS], fontsize=5.5)
    ax.set_ylabel("Ranking stability")
    ax.set_xlim(-0.5, len(SCENARIOS)-0.5)
    ax.legend(loc="lower left", fontsize=5, handlelength=1.0)
    print(f"  (c) rho={C['spearman'].median():.3f}, "
          f"overlap={C['overlap'].median():.2f}")

lgrid(ax); plabel(ax,"c")

for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS9_robustness.{ext}")
plt.show()


In [ ]:
# FIG S10 - Planning Resolution Flexibility (1x3 maps)
# Drawn by figures/figS10_huc_flexibility/make_figure.py: control points from the HUC run (947/158/25),
# percentages computed from the run summary, colorbar "Peak reduction lost under outlet control (pp)".
# It reads shape/, huc_spatial_compact.csv and outputs_huc_control/ under FAC_DATA_ROOT and writes
# figS10_huc_flexibility.{pdf,png} to FAC_OUT_ROOT/figS10_huc_flexibility/ (see paths.py).
print("\nFIG S10 - Planning Resolution Flexibility")
import subprocess, sys
r = subprocess.run([sys.executable, str(REPO / "figures" / "figS10_huc_flexibility" / "make_figure.py")],
                   capture_output=True, text=True)
print(r.stdout, r.stderr)
r.check_returncode()

In [ ]:
# FIG S11 - Direction and Magnitude of Lifted Inputs (1x3) [delta-based a,b]
print("\nFIG S11 - Lifted inputs")

DELTA_DIR = DATA / "outputs_flood_rl2"
delta_parquet = DELTA_DIR / "delta_atten_per_reach.parquet"
has_delta = delta_parquet.exists()
has_feas  = "feas_median_mean_ratio_flow" in df_mass.columns

if has_delta:
    df_delta = pd.read_parquet(delta_parquet)
    print(f"  Loaded delta data: {len(df_delta):,} rows")

fig, axes = plt.subplots(1, 3, figsize=(W_FULL, 70*MM),
                          gridspec_kw={"wspace": 0.34})

# (a) Histogram: delta atten fraction during exceedance
ax = axes[0]
if has_delta:
    v = df_delta["delta_atten_flood_frac"].dropna().values
    v = v[np.isfinite(v) & (v >= 0) & (v <= 1)] * 100

    ax.hist(v, bins=60, density=True, color="#9ecae1", alpha=0.85,
            edgecolor="#4292c6", linewidth=0.3, zorder=2)

    med = np.median(v)
    ax.axvline(med, color="#333", ls="--", lw=0.9, zorder=5)
    yl = ax.get_ylim()[1]
    ax.text(med - 4, yl * 0.93, f"median\n{med:.0f}%", fontsize=6,
            fontweight="bold", ha="right", va="top")

    ax.set_xlabel(r"Attenuation fraction $|\delta_i^-|/|\delta_i|$"
                  "\nduring exceedance (%)")
    ax.set_ylabel("Density"); ax.set_xlim(0, 105)
    print(f"  (a) delta flood atten: median={med:.1f}%")
else:
    ax.text(0.5, 0.5, "delta data not available\nRun analyses/attenuation_direction/\ncompute_delta_atten_fraction.py",
            transform=ax.transAxes, ha="center", fontsize=6)
lgrid(ax); plabel(ax, "a")

# (b) Basin-scale delta attenuation per SSP
ax = axes[1]
if has_delta:
    rows_b = []
    for (gcm, hydro, ssp), g in df_delta.groupby(["gcm", "hydro", "ssp"]):
        atten = g["delta_atten_L1"].sum()
        total = g["delta_total_L1"].sum()
        rows_b.append({"ssp": int(ssp), "val": atten / (total + 1e-12) * 100})
    dfc = pd.DataFrame(rows_b)

    for i, scn in enumerate(SCENARIOS):
        vals = dfc[dfc["ssp"] == scn]["val"].values
        if len(vals) == 0: continue
        jitter_strip(ax, i, vals, SSP_COLORS[scn], ms=4, alpha=0.55)
        med = np.median(vals)
        q25, q75 = np.percentile(vals, [25, 75])
        ax.plot([i, i], [q25, q75], color=SSP_COLORS[scn], lw=2.0,
                solid_capstyle="round", zorder=4)
        ax.plot(i, med, "o", color=SSP_COLORS[scn], ms=7,
                markeredgecolor="black", markeredgewidth=0.5, zorder=5)
        ax.text(i, q25 - 4, f"{med:.1f}%", ha="center", fontsize=5.5,
                fontweight="bold", va="top")
        print(f"  (b) {SSP_LABELS[scn]}: {med:.1f}% [{q25:.1f}-{q75:.1f}]")

    all_vr = dfc["val"].values
    ax.text(0.97, 0.05, f"all: {np.median(all_vr):.1f}%\n(n={len(all_vr)})",
            transform=ax.transAxes, fontsize=6.5, va="bottom", ha="right",
            bbox=dict(boxstyle="round,pad=0.15", fc="white",
                      ec="none", lw=0.3, alpha=0.85))
    ax.set_xticks(range(len(SCENARIOS)))
    ax.set_xticklabels([SSP_LABELS[s] for s in SCENARIOS], fontsize=6)
    ax.set_ylabel("Basin-scale attenuation\nfraction (%)")
    ax.set_xlim(-0.5, len(SCENARIOS)-0.5)
    ax.set_ylim(0, 102)
else:
    ax.text(0.5, 0.5, "delta data not available",
            transform=ax.transAxes, ha="center", fontsize=6)
lgrid(ax); plabel(ax, "b")

# (c) |u_i|/x_i during exceedance (median only)
ax = axes[2]
if has_feas:
    for i, scn in enumerate(SCENARIOS):
        sub = df_mass[df_mass["ssp"]==scn]
        vals_med = sub["feas_median_mean_ratio_flow"].dropna().values * 100

        med  = np.median(vals_med)
        q25, q75 = np.percentile(vals_med, [25, 75])
        ax.plot([i, i], [q25, q75], color=SSP_COLORS[scn], lw=2.0,
                solid_capstyle="round", zorder=3)
        ax.plot(i, med, "o", color=SSP_COLORS[scn], ms=7,
                markeredgecolor="black", markeredgewidth=0.6, zorder=4)
        ax.text(i, q75 + 2.0, f"{med:.0f}%", ha="center", fontsize=5.5,
                fontweight="bold")
        print(f"  (c) {SSP_LABELS[scn]}: median={med:.1f}%")

    all_med = df_mass["feas_median_mean_ratio_flow"].dropna().values * 100
    ax.text(0.97, 0.95,
            f"all: median={np.median(all_med):.0f}%\n(n={len(all_med)})",
            transform=ax.transAxes, fontsize=6.5, va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.15", fc="white",
                      ec="none", lw=0.3, alpha=0.85))
    ax.set_xticks(range(len(SCENARIOS)))
    ax.set_xticklabels([SSP_LABELS[s] for s in SCENARIOS], fontsize=6)
    ax.set_ylabel(r"$|u_i|/x_i$ during exceedance (%)")
    ax.set_xlim(-0.5, len(SCENARIOS)-0.5)
    ax.set_ylim(0, 30)
    print(f"  (c) Pooled: median={np.median(all_med):.1f}%")
else:
    ax.text(0.5, 0.5, "Feasibility data\nnot available",
            transform=ax.transAxes, ha="center", fontsize=7)
lgrid(ax); plabel(ax, "c")

for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"figS11_lifted_inputs.{ext}")
plt.show()

print("\n" + "="*60)
print("  ALL DONE - saved to", FIG_DIR)
print("  S4: figS4_extreme_flows.pdf")
print("  S5: figS5_mass_fidelity.pdf")
print("  S7: figS7_effort_scaling.pdf")
print("  S8: figS8_stream_order.pdf")
print("  S9: figS9_robustness.pdf")
print("  S10: figS10_huc_flexibility.pdf (in FAC_OUT_ROOT/figS10_huc_flexibility/)")
print("  S11: figS11_lifted_inputs.pdf")
print("="*60)
